#1. Data cleaning and Labelling


In [ ]:
# Mount the drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


#A. Car and Motorcycle data collection

**A1. Unzip the folder**

In [ ]:
zip_path = "/content/drive/MyDrive/EE655_Project/HeteroTraffic Annotated Dataset for Multi-Class Ve.zip"

In [ ]:
import zipfile

extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Outer zip extracted!")

Outer zip extracted!


**A1.a. Unzip inner zipped folder**

In [ ]:
inner_zip = "/content/dataset/HeteroTraffic Annotated Dataset for Multi-Class Ve/roadvision_blur.zip"

extract_path_inner = "/content/dataset/roadvision_blur"

with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_path_inner)

print("Inner zip extracted!")

Inner zip extracted!


**A2. Find the structure of folder**

In [ ]:
import os

print(os.listdir("/content/dataset/roadvision_blur"))

['roadvision_blur']


In [ ]:
print(os.listdir("/content/dataset/roadvision_blur/roadvision_blur"))

['images', 'Classes.txt', 'data.yaml', 'AnnotationsDetails.txt', 'labels']


**A3. Prepare folder to upload cleaned dataset**

In [ ]:
# INPUT (your dataset)
dataset_path = "/content/dataset/roadvision_blur/roadvision_blur"

# OUTPUT (you choose this folder in Drive)
output_path = "/content/drive/MyDrive/EE655_Project/Cleaned_Datase_Ve_Detection"

In [ ]:
import os

os.makedirs(f"{output_path}/images", exist_ok=True)
os.makedirs(f"{output_path}/labels", exist_ok=True)

**A4. Class selection for car and motorcycle**

In [ ]:
with open(f"{dataset_path}/Classes.txt") as f:
    classes = f.read().splitlines()

print(classes)

['Bicycle', 'Bus', 'Bhotbhoti', 'Car', 'CNG', 'Easybike', 'Leguna', 'Motorbike', 'MPV', 'Pedestrian', 'Pickup\t', 'PowerTiller', 'Rickshaw', 'ShoppingVan', 'Truck', 'Van', 'Wheelbarrow']


In [ ]:
car_id = classes.index('Car')
bike_id = classes.index('Motorbike')

print("Car ID:", car_id)
print("Motorbike ID:", bike_id)

Car ID: 3
Motorbike ID: 7


**A5. Data extraction and save in drive**

In [ ]:
import os
import shutil

labels_path = f"{dataset_path}/labels"
images_path = f"{dataset_path}/images"

out_labels = f"{output_path}/labels"
out_images = f"{output_path}/images"

count = 0

for file in os.listdir(labels_path):

    label_file = os.path.join(labels_path, file)

    with open(label_file, "r") as f:
        lines = f.readlines()

    new_lines = []

    for line in lines:
        parts = line.strip().split()
        cls = int(parts[0])

        # Keep only Car & Motorcycle
        if cls == car_id:
            parts[0] = "0"
            new_lines.append(" ".join(parts))

        elif cls == bike_id:
            parts[0] = "1"
            new_lines.append(" ".join(parts))

    # Save only useful images
    if len(new_lines) > 0:

        # Save label
        with open(os.path.join(out_labels, file), "w") as f:
            f.write("\n".join(new_lines))

        # Copy image
        img_name = file.replace(".txt", ".jpg")  # adjust if needed
        src_img = os.path.join(images_path, img_name)
        dst_img = os.path.join(out_images, img_name)

        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)
            count += 1

print("✅ Total extracted images:", count)

✅ Total extracted images: 7353


#B. Seatbelt Data cleaning

**B1. Unzip the dataset**

In [ ]:
zip_path = "/content/drive/MyDrive/EE655_Project/Seatbelt Detection.v3i.yolov8.zip"

In [ ]:
import zipfile
import os

extract_path = "/content/new_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted!")

✅ Dataset extracted!


**B2. Find the structure and original classes**

In [ ]:
print(os.listdir(extract_path))

['README.roboflow.txt', 'valid', 'test', 'train', 'README.dataset.txt', 'data.yaml']


In [ ]:
print(os.listdir(f"{extract_path}/train"))

['images', 'labels']


In [ ]:
import yaml

with open(f"{extract_path}/data.yaml") as f:
    data = yaml.safe_load(f)

classes = data['names']

print("Original Classes:", classes)

Original Classes: ['person-noseatbelt', 'person-seatbelt', 'seatbelt', 'windshield']


**B2.a. Convert original classes into two required classes**

In [ ]:
# Keep only these two classes
map_dict = {
    'person-seatbelt': 'Seatbelt',
    'person-noseatbelt': 'NoSeatbelt'
}

# Assign new class IDs
new_class_map = {
    'Seatbelt': 0,
    'NoSeatbelt': 1
}

**B3. Prepare storing folder**

In [ ]:
import os

output_path = "/content/drive/MyDrive/EE655_Project/Cleaned_Dataset_Seatbelt_Detection"

splits = ['train', 'valid', 'test']

for split in splits:
    os.makedirs(f"{output_path}/{split}/images", exist_ok=True)
    os.makedirs(f"{output_path}/{split}/labels", exist_ok=True)

print("✅ Output folders created")

✅ Output folders created


**B4. Clean, convert and store the dataset**

In [ ]:
import os
import shutil

total_images = 0

for split in splits:

    print(f"\nProcessing {split}...")

    labels_path = f"{extract_path}/{split}/labels"
    images_path = f"{extract_path}/{split}/images"

    out_labels = f"{output_path}/{split}/labels"
    out_images = f"{output_path}/{split}/images"

    for file in os.listdir(labels_path):

        label_file = os.path.join(labels_path, file)

        with open(label_file) as f:
            lines = f.readlines()

        new_lines = []

        # Process each object
        for line in lines:
            parts = line.strip().split()
            cls = int(parts[0])

            class_name = classes[cls]

            # Keep only required classes
            if class_name in map_dict:

                new_class = map_dict[class_name]
                new_id = new_class_map[new_class]

                parts[0] = str(new_id)
                new_lines.append(" ".join(parts))

        # Save only useful images
        if len(new_lines) > 0:

            # Save label
            with open(os.path.join(out_labels, file), "w") as f:
                f.write("\n".join(new_lines))

            # Copy image
            img_name = file.replace(".txt", ".jpg")  # change if png
            src_img = os.path.join(images_path, img_name)
            dst_img = os.path.join(out_images, img_name)

            if os.path.exists(src_img):
                shutil.copy(src_img, dst_img)
                total_images += 1

print("\n✅ Cleaning Complete!")
print("Total images kept:", total_images)


Processing train...

Processing valid...

Processing test...

✅ Cleaning Complete!
Total images kept: 5272


#C.Helmet Dataset


**C1. Extract the zip file**

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/EE655_Project/Motorcycle Helmet Detection.v2i.yolov8.zip"  #

extract_path = "/content/helmet_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted!")

✅ Dataset extracted!


**C2. Find out structure and classes and convert them into desired ones**

In [ ]:
import os

print(os.listdir(extract_path))

['README.roboflow.txt', 'valid', 'test', 'train', 'README.dataset.txt', 'data.yaml']


In [ ]:
import yaml

with open(f"{extract_path}/data.yaml") as f:
    data = yaml.safe_load(f)

classes = data['names']
print("Original Classes:", classes)

Original Classes: ['full-faced', 'half-faced', 'invalid', 'no helmet']


In [ ]:
map_dict = {
    'full-faced': 'Helmet',
    'half-faced': 'Helmet',
    'no helmet': 'NoHelmet'
}

new_class_map = {
    'Helmet': 0,
    'NoHelmet': 1
}

**C3. Create storage folder**

In [ ]:
output_path = "/content/drive/MyDrive/EE655_Project/Cleaned_Helmet_Dataset"

splits = ['train', 'valid', 'test']

import os

for split in splits:
    os.makedirs(f"{output_path}/{split}/images", exist_ok=True)
    os.makedirs(f"{output_path}/{split}/labels", exist_ok=True)

print("✅ Output folders ready")

✅ Output folders ready


**C4. Clean, convert and store the new dataset**

In [ ]:
import os
import shutil

total_images = 0

for split in splits:

    print(f"\nProcessing {split}...")

    labels_path = f"{extract_path}/{split}/labels"
    images_path = f"{extract_path}/{split}/images"

    out_labels = f"{output_path}/{split}/labels"
    out_images = f"{output_path}/{split}/images"

    for file in os.listdir(labels_path):

        label_file = os.path.join(labels_path, file)

        with open(label_file) as f:
            lines = f.readlines()

        new_lines = []

        for line in lines:
            parts = line.strip().split()
            cls = int(parts[0])

            class_name = classes[cls]

            # Keep only required classes
            if class_name in map_dict:

                new_class = map_dict[class_name]
                new_id = new_class_map[new_class]

                parts[0] = str(new_id)
                new_lines.append(" ".join(parts))

        # Save only useful images
        if len(new_lines) > 0:

            # Save label
            with open(os.path.join(out_labels, file), "w") as f:
                f.write("\n".join(new_lines))

            # Copy image
            img_name = file.replace(".txt", ".jpg")
            src_img = os.path.join(images_path, img_name)
            dst_img = os.path.join(out_images, img_name)

            if os.path.exists(src_img):
                shutil.copy(src_img, dst_img)
                total_images += 1

print("\n✅ Cleaning Complete!")
print("Total images kept:", total_images)


Processing train...

Processing valid...

Processing test...

✅ Cleaning Complete!
Total images kept: 2111


#D. Split the Vehicle Detection data

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# INPUT (your cleaned dataset)
input_path = "/content/drive/MyDrive/EE655_Project/Cleaned_Datase_Ve_Detection"

images_path = f"{input_path}/images"
labels_path = f"{input_path}/labels"

# OUTPUT
output_path = "/content/drive/MyDrive/EE655_Project/Cleaned_Splited_Ve_Data"

In [ ]:
splits = ['train', 'val', 'test']

for split in splits:
    os.makedirs(f"{output_path}/images/{split}", exist_ok=True)
    os.makedirs(f"{output_path}/labels/{split}", exist_ok=True)

print("✅ Folders created")

✅ Folders created


In [ ]:
images = os.listdir(images_path)

print("Total images:", len(images))

Total images: 7353


In [ ]:
# First split: train (70%) and temp (30%)
train_imgs, temp_imgs = train_test_split(images, test_size=0.3, random_state=42)

# Second split: val (20%) and test (10%)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.33, random_state=42)

print("Train:", len(train_imgs))
print("Val:", len(val_imgs))
print("Test:", len(test_imgs))

Train: 5147
Val: 1478
Test: 728


In [ ]:
import os
import shutil

# -------------------------------
# Function to move files with progress
# -------------------------------
def move_files(file_list, split):

    total = len(file_list)
    count = 0

    for img in file_list:
        label = img.replace(".jpg", ".txt")  # adjust if png

        src_img = f"{images_path}/{img}"
        src_lbl = f"{labels_path}/{label}"

        dst_img = f"{output_path}/images/{split}/{img}"
        dst_lbl = f"{output_path}/labels/{split}/{label}"

        if os.path.exists(src_img) and os.path.exists(src_lbl):
            shutil.copy(src_img, dst_img)
            shutil.copy(src_lbl, dst_lbl)
            count += 1

        # 🔥 Print progress every 200 files
        if count % 200 == 0:
            print(f"{split.upper()} Progress: {count}/{total}")

    print(f"✅ {split.upper()} completed: {count} files copied\n")


# -------------------------------
# Run splitting
# -------------------------------
move_files(train_imgs, "train")
move_files(val_imgs, "val")
move_files(test_imgs, "test")

# -------------------------------
# Final verification
# -------------------------------
print("📊 FINAL COUNTS:")
print("Train images:", len(os.listdir(f"{output_path}/images/train")))
print("Val images:", len(os.listdir(f"{output_path}/images/val")))
print("Test images:", len(os.listdir(f"{output_path}/images/test")))

TRAIN Progress: 200/5147
TRAIN Progress: 400/5147
TRAIN Progress: 600/5147
TRAIN Progress: 800/5147
TRAIN Progress: 1000/5147
TRAIN Progress: 1200/5147
TRAIN Progress: 1400/5147
TRAIN Progress: 1600/5147
TRAIN Progress: 1800/5147
TRAIN Progress: 2000/5147
TRAIN Progress: 2200/5147
TRAIN Progress: 2400/5147
TRAIN Progress: 2600/5147
TRAIN Progress: 2800/5147
TRAIN Progress: 3000/5147
TRAIN Progress: 3200/5147
TRAIN Progress: 3400/5147
TRAIN Progress: 3600/5147
TRAIN Progress: 3800/5147
TRAIN Progress: 4000/5147
TRAIN Progress: 4200/5147
TRAIN Progress: 4400/5147
TRAIN Progress: 4600/5147
TRAIN Progress: 4800/5147
TRAIN Progress: 5000/5147
✅ TRAIN completed: 5147 files copied

VAL Progress: 200/1478
VAL Progress: 400/1478
VAL Progress: 600/1478
VAL Progress: 800/1478
VAL Progress: 1000/1478
VAL Progress: 1200/1478
VAL Progress: 1400/1478
✅ VAL completed: 1478 files copied

TEST Progress: 200/728
TEST Progress: 400/728
TEST Progress: 600/728
✅ TEST completed: 728 files copied

📊 FINAL COU

#Reduce and Unzip Lisence plate data


In [ ]:
zip_path = "/content/drive/MyDrive/EE655_Project/License Plate Recognition.v13i.yolov8.zip"

In [ ]:
import zipfile
import os

extract_path = "/content/license_plate_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted successfully!")

✅ Dataset extracted successfully!


In [ ]:
import os
import shutil
import random

base_path = extract_path  # original dataset
output_path = "/content/drive/MyDrive/EE655_Project/Number_Plate_Detection"

In [ ]:
splits = ['train', 'valid', 'test']

for split in splits:
    os.makedirs(f"{output_path}/{split}/images", exist_ok=True)
    os.makedirs(f"{output_path}/{split}/labels", exist_ok=True)

print("✅ Output folders ready")

✅ Output folders ready


In [ ]:
train_images_path = f"{base_path}/train/images"
train_labels_path = f"{base_path}/train/labels"

all_images = os.listdir(train_images_path)

print("Original train:", len(all_images))

selected_images = random.sample(all_images, 10000)

print("Selected:", len(selected_images))

Original train: 98798
Selected: 10000


In [ ]:
for img in selected_images:
    label = img.replace(".jpg", ".txt")  # adjust if png

    src_img = f"{train_images_path}/{img}"
    src_lbl = f"{train_labels_path}/{label}"

    dst_img = f"{output_path}/train/images/{img}"
    dst_lbl = f"{output_path}/train/labels/{label}"

    if os.path.exists(src_img) and os.path.exists(src_lbl):
        shutil.copy(src_img, dst_img)
        shutil.copy(src_lbl, dst_lbl)

print("✅ Train reduced to 10k")

✅ Train reduced to 10k


In [ ]:
shutil.copytree(
    f"{base_path}/valid",
    f"{output_path}/valid",
    dirs_exist_ok=True
)

print("✅ Validation copied")

✅ Validation copied


In [ ]:
shutil.copytree(
    f"{base_path}/test",
    f"{output_path}/test",
    dirs_exist_ok=True
)

print("✅ Test copied")

✅ Test copied


In [ ]:
print("📊 FINAL DATASET")

print("Train:", len(os.listdir(f"{output_path}/train/images")))
print("Val:", len(os.listdir(f"{output_path}/valid/images")))
print("Test:", len(os.listdir(f"{output_path}/test/images")))

📊 FINAL DATASET
Train: 10000
Val: 2048
Test: 1020


In [ ]:
import shutil
import os

if os.path.exists(f"{output_path}/valid"):
    shutil.rmtree(f"{output_path}/valid")

if os.path.exists(f"{output_path}/test"):
    shutil.rmtree(f"{output_path}/test")

print("✅ Old empty folders removed")

✅ Old empty folders removed


In [ ]:
shutil.copytree(f"{base_path}/valid", f"{output_path}/valid")
shutil.copytree(f"{base_path}/test", f"{output_path}/test")

print("✅ Valid & Test copied successfully!")

In [ ]:
print("Train:", len(os.listdir(f"{output_path}/train/images")))
print("Val:", len(os.listdir(f"{output_path}/valid/images")))
print("Test:", len(os.listdir(f"{output_path}/test/images")))

Train: 10000
Val: 2048
Test: 1020
